<h1><font color="#113D68" size=6>Deep Learning en Python con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 4. Mejorar la Generalización en modelos</font></h1>

<h1><font color="#113D68" size=4>8. Optimización de hiperparámetros con GridSearch
</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. Modelos de PyTorch en scikit-learn](#section1)
* [2. Descripción del Problema](#section2)
* [3. Grid Search en scikit-learn](#section3)
* [4. Tamaño de Lote y el Número de Épocas](#section4)
* [5. Ajustar el Algoritmo de Optimización del Entrenamiento](#section5)
* [6. Ajustar la Tasa de Aprendizaje y el Momentum](#section6)
* [7. Ajustar la Inicialización de Pesos de la Red](#section7)
* [8. Ajustar la Función de Activación de las Neuronas](#section8)
* [9. Ajustar la Regularización Dropout](#section9)
* [10. Ajustar el Número de Neuronas en la Capa Oculta](#section10)
* [11. Consejos para la Optimización de Hiperparámetros](#section11)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

Los **pesos** de una red neuronal se denominan "parámetros" en el código de `PyTorch` y son ajustados por el optimizador durante el entrenamiento. En cambio, los **hiperparámetros** son los parámetros de una red neuronal que se fijan por diseño y no se ajustan durante el entrenamiento. Ejemplos de hiperparámetros incluyen el número de capas ocultas y la elección de las funciones de activación. La optimización de hiperparámetros es una parte crucial en el aprendizaje profundo, ya que las redes neuronales son notoriamente difíciles de configurar y requieren ajustar muchos parámetros. Además, entrenar modelos individuales puede ser muy lento.

En esta clase, descubrirás cómo utilizar la capacidad de búsqueda en cuadrícula (*Grid Search*) de la biblioteca Scikit-learn en Python para ajustar los hiperparámetros de modelos de aprendizaje profundo en PyTorch. Después de leer este capítulo, sabrás:

- Cómo envolver modelos de PyTorch para usarlos en scikit-learn y cómo aplicar grid search.
- Cómo hacer grid search en parámetros comunes de redes neuronales, como la tasa de aprendizaje, tasa de dropout, épocas de entrenamiento, y el número de neuronas.
- Cómo definir tus propios experimentos de ajuste de hiperparámetros en tus proyectos.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch tutorials.](https://pytorch.org/tutorials/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch API: [`torch.nn.init`](https://pytorch.org/docs/stable/nn.init.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch API: [`torch.nn`](https://pytorch.org/docs/stable/nn.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch API: [`torch.optim`](https://pytorch.org/docs/stable/optim.html)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Modelos de PyTorch en scikit-learn</font>

- Integración de PyTorch con scikit-learn:
  - Uso de `skorch` como envoltorio

  - Proporciona una API similar a scikit-learn para modelos PyTorch

- Clases principales de skorch:
  - `NeuralNetClassifier`: Para redes neuronales de clasificación

  - `NeuralNetRegressor`: Para redes neuronales de regresión

- Ventajas:
  - Permite usar modelos PyTorch con funcionalidades de scikit-learn

  - Facilita el uso de validación cruzada, búsqueda de hiperparámetros, etc.

- Consideraciones:
  - Requiere que los modelos PyTorch se adapten ligeramente para trabajar con skorch
  
  - Puede requerir ajustes en la forma de especificar hiperparámetros

In [1]:
!pip install -q skorch  # Colab no lo trae instalado

Si estás usando `conda`, `skorch` solo está disponible en el canal `forge`:

```bash
    conda install -c conda-forge pytorch skorch scikit-learn
```

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`skorch`](https://skorch.readthedocs.io/en/stable/)

Para usar estos envoltorios:

1. Primero debes definir tu modelo de PyTorch como una clase usando `nn.Module`, y 

2. Luego pasar el nombre de la clase al argumento `module` al construir la clase `NeuralNetClassifier`.

In [2]:
import torch.nn as nn
from skorch import NeuralNetClassifier

class SonarClassifier(nn.Module):
    def __init__(self, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(60, 60) for _ in range(n_layers)])  # ModuleList registra los parámetros
        self.act = nn.ReLU()
        self.output = nn.Linear(60, 1)
        self.prob = nn.Sigmoid()

    def forward(self, x):
        for layer in self.layers:
            x = self.act(layer(x))
        x = self.prob(self.output(x))
        return x

Este enfoque permite una configuración inicial completa del modelo, simplificando el proceso de entrenamiento y ajuste.

- Constructor de `NeuralNetClassifier`:
  - Puede tomar argumentos predeterminados para `model.fit()`

  - Puede tomar argumentos adicionales para el constructor del modelo con el el prefijo `module__` (con dos guiones bajos)

  - Permite pasar parámetros directamente al modelo PyTorch
  
  - Sobrescribe valores predeterminados en el constructor del modelo

  - Ejemplos de argumentos:

    - Número de épocas

    - Tamaño del lote

- Argumentos comunes:
  - `module`: Clase o instancia del módulo PyTorch

  - `criterion`: Función de pérdida (por defecto `torch.nn.NLLLoss`)

  - `optimizer`: Optimizador (por defecto `torch.optim.SGD`)

  - `lr`: Tasa de aprendizaje

  - `max_epochs`: Número máximo de épocas de entrenamiento

  - `batch_size`: Tamaño del lote para entrenamiento

- Flexibilidad en la configuración:
  - Permite establecer parámetros de entrenamiento al crear el modelo

  - Facilita la personalización del proceso de entrenamiento

- Consideraciones:
  - Los parámetros pueden ser ajustados posteriormente usando `set_params()`

  - Asegurarse de usar el prefijo correcto (`module__`)
  
  - Facilita la búsqueda de hiperparámetros con herramientas de scikit-learn



In [3]:
model = NeuralNetClassifier(
    module=SonarClassifier,
    criterion=nn.BCELoss,
    max_epochs=150,
    batch_size=10,
    module__n_layers=2  # module__<arg> se pasa al constructor del modelo
)

In [4]:
print(model.initialize())

<class 'skorch.classifier.NeuralNetClassifier'>[initialized](
  module_=SonarClassifier(
    (layers): ModuleList(
      (0-1): 2 x Linear(in_features=60, out_features=60, bias=True)
    )
    (act): ReLU()
    (output): Linear(in_features=60, out_features=1, bias=True)
    (prob): Sigmoid()
  ),
)


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. Descripción del Problema</font>

- Exploración de ejemplos:
  - Uso de modelos PyTorch con Scikit-learn

  - Aplicación de Grid search en scikit-learn

- Conjunto de datos utilizado:
  - Nombre: Pima Indians Onset of Diabetes Classification Dataset

  - Características:
    - Pequeño conjunto de datos

    - Atributos numéricos

    - Fácil de manipular y trabajar

- Objetivo:
  - Demostrar el proceso de optimización de modelos PyTorch usando herramientas de scikit-learn
  
  - Ilustrar la flexibilidad de la integración entre PyTorch y scikit-learn

- Consideraciones:
  - Los resultados pueden no ser óptimos globalmente

  - Útil para entender el impacto individual de cada parámetro

- Estructura de los ejemplos:
  1. Introducción al conjunto de datos

  2. Implementación básica del modelo

  3. Aplicación de Grid search para diferentes parámetros

  4. Análisis de resultados en cada paso


In [5]:
# Descarga de datos (necesario en Colab): se guardan en ./Datasets
import os, urllib.request

os.makedirs('Datasets', exist_ok=True)
urls = {
    'pima-indians-diabetes.csv':
        'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv',
}
for nombre, url in urls.items():
    if not os.path.exists(f'Datasets/{nombre}'):
        urllib.request.urlretrieve(url, f'Datasets/{nombre}')

In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from skorch import NeuralNetClassifier
from sklearn.model_selection import GridSearchCV

dataset = np.loadtxt('Datasets/pima-indians-diabetes.csv', delimiter=',')
X = dataset[:,0:8]
y = dataset[:,8]
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

Creamos ahora la red neuronal de línea base

In [7]:
class PimaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(8, 12)
        self.act = nn.ReLU()
        self.output = nn.Linear(12, 1)
        self.prob = nn.Sigmoid()

    def forward(self, x):
        x = self.act(self.layer(x))
        x = self.prob(self.output(x))
        return x

model = NeuralNetClassifier(
    PimaClassifier,
    criterion=nn.BCELoss,
    optimizer=optim.Adam,
    max_epochs=100,
    batch_size=10,
    verbose=False
)

print(model.initialize())

<class 'skorch.classifier.NeuralNetClassifier'>[initialized](
  module_=PimaClassifier(
    (layer): Linear(in_features=8, out_features=12, bias=True)
    (act): ReLU()
    (output): Linear(in_features=12, out_features=1, bias=True)
    (prob): Sigmoid()
  ),
)


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. Grid Search en scikit-learn</font>

Este enfoque proporciona una manera sistemática y completa de optimizar hiperparámetros en modelos de aprendizaje automático, incluyendo redes neuronales implementadas con PyTorch y skorch.

- Grid Search:
  - Técnica de optimización de hiperparámetros

  - Prueba todas las combinaciones posibles de hiperparámetros

  - Encuentra la combinación que proporciona el mejor puntaje

- Implementación en Scikit-learn:

  - Clase: `GridSearchCV`

- Configuración de `GridSearchCV`:

  - Argumento `param_grid`: Diccionario de hiperparámetros a evaluar

    - Clave: Nombre del parámetro del modelo

    - Valor: Arreglo de valores a probar

- Métrica de optimización:

  - Por defecto: Accuracy

  - Personalizable: Usar argumento `scoring` en el constructor

- Configuración de validación cruzada:

  - Por defecto: 3-fold cross-validation

  - Personalizable: Usar argumento `cv` en el constructor

- Ventajas:
  - Búsqueda exhaustiva de hiperparámetros

  - Fácil integración con modelos de scikit-learn y skorch

- Consideraciones:
  - Puede ser computacionalmente costoso para grandes espacios de parámetros
  
  - Importante balancear exhaustividad con tiempo de cómputo disponible



<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

In [8]:
# modelo NUEVO sin inicializar: un estimador ya inicializado (model.initialize()) no se puede
# enviar a los procesos paralelos de n_jobs=-1
model = NeuralNetClassifier(
    PimaClassifier,
    criterion=nn.BCELoss,
    optimizer=optim.Adam,
    verbose=False
)

param_grid = {
    'batch_size': [10, 20, 40, 60, 80, 100],
    'max_epochs': [10, 50, 100]
}
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X, y)

- Optimización del rendimiento de GridSearchCV:

  - Argumento `n_jobs=-1` en el constructor

  - Utiliza todos los núcleos de la máquina

  - Aprovecha CPUs con múltiples núcleos

  - Sin especificar: Ejecución en un solo hilo (más lento en procesadores multicore)

- Acceso a los resultados del Grid Search:

  - Objeto: `grid_result` (devuelto por `grid.fit()`)

- Atributos importantes del resultado:
  1. `best_score_`:
     - Proporciona el mejor puntaje observado

     - Refleja el rendimiento óptimo alcanzado

  2. `best_params_`:
     - Describe la combinación de parámetros óptima

     - Muestra los hiperparámetros que lograron el mejor resultado

Este enfoque permite una búsqueda de hiperparámetros eficiente y una fácil interpretación de los resultados, optimizando el proceso de ajuste de modelos de aprendizaje automático.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section4"></a>
# <font color="#004D7F" size=6>4. Tamaño de Lote y el Número de Épocas</font>

Este ejemplo demuestra cómo realizar una búsqueda sistemática de los parámetros fundamentales de entrenamiento en redes neuronales, sentando las bases para optimizaciones más avanzadas.

- Ajuste de hiperparámetros en redes neuronales:
  1. Tamaño de lote (batch size)

  2. Número de épocas

- Tamaño de lote (batch size):
  - Definición: Cantidad de patrones mostrados antes de actualizar pesos

  - Función: Optimización en el entrenamiento

  - Impacto: Define cuántos patrones se leen y mantienen en memoria simultáneamente

  - Relevancia: Especialmente importante en redes LSTM y convolucionales

- Número de épocas:
  - Definición: Cuántas veces se muestra el conjunto de datos completo a la red

  - Impacto: Afecta el tiempo de entrenamiento y la convergencia del modelo

- Ejemplo de evaluación:
  - Rango de tamaños de minibatch: 10 a 100

  - Incrementos: Pasos de 20

- Consideraciones:
  - El tamaño de lote óptimo puede variar según la arquitectura de la red

  - Un número adecuado de épocas es crucial para evitar subajuste o sobreajuste

- Ventajas de este enfoque:
  - Permite encontrar la combinación óptima de tamaño de lote y épocas

  - Ayuda a balancear eficiencia de entrenamiento y rendimiento del modelo

- Aplicación:
  - Útil para optimizar el rendimiento básico del modelo
  
  - Proporciona una base para ajustes más finos en pasos posteriores


In [9]:
print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Mejor: 0.722656 usando {'batch_size': 10, 'max_epochs': 100}
0.674479 (0.009744) with: {'batch_size': 10, 'max_epochs': 10}
0.710938 (0.041829) with: {'batch_size': 10, 'max_epochs': 50}
0.722656 (0.022999) with: {'batch_size': 10, 'max_epochs': 100}
0.669271 (0.012890) with: {'batch_size': 20, 'max_epochs': 10}
0.699219 (0.025315) with: {'batch_size': 20, 'max_epochs': 50}
0.709635 (0.009744) with: {'batch_size': 20, 'max_epochs': 100}
0.658854 (0.027126) with: {'batch_size': 40, 'max_epochs': 10}
0.708333 (0.018688) with: {'batch_size': 40, 'max_epochs': 50}
0.701823 (0.037377) with: {'batch_size': 40, 'max_epochs': 100}
0.667969 (0.030425) with: {'batch_size': 60, 'max_epochs': 10}
0.690104 (0.024360) with: {'batch_size': 60, 'max_epochs': 50}
0.699219 (0.033299) with: {'batch_size': 60, 'max_epochs': 100}
0.684896 (0.032734) with: {'batch_size': 80, 'max_epochs': 10}
0.694010 (0.006639) with: {'batch_size': 80, 'max_epochs': 50}
0.705729 (0.033502) with: {'batch_size': 80, 'max_epo

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section5"></a>
# <font color="#004D7F" size=6>5. Ajustar el Algoritmo de Optimización del Entrenamiento</font>

Este ejemplo permite una comparación rápida de diferentes optimizadores en PyTorch, proporcionando una base para una exploración más detallada de los algoritmos más prometedores para un problema específico.

- Ajuste del algoritmo de optimización:
  - Se evaluarán diferentes optimizadores

  - Cada uno con sus parámetros predeterminados

- Enfoque inusual:
  - Normalmente se elige un optimizador a priori

  - Luego se ajustan sus parámetros específicos

- Optimizadores a evaluar:
  - SGD (Stochastic Gradient Descent)

  - RMSprop

  - Adam

  - Adadelta

  - Adagrad

  - Adamax

  - ASGD (Averaged Stochastic Gradient Descent)

- Consideraciones:
  - Cada optimizador puede tener un rendimiento diferente según el problema

  - Los parámetros predeterminados pueden no ser óptimos para todos los casos

- Ventajas de este enfoque:
  - Proporciona una visión general del rendimiento de diferentes optimizadores

  - Puede ayudar a identificar optimizadores prometedores para un ajuste más fino

- Limitaciones:
  - No considera el ajuste fino de cada optimizador
  
  - Los resultados pueden no ser directamente aplicables a todos los problemas



In [10]:
model = NeuralNetClassifier(
    PimaClassifier,
    criterion=nn.BCELoss,
    max_epochs=100,
    batch_size=10,
    verbose=False
)

param_grid = {
    'optimizer': [optim.SGD, optim.RMSprop, optim.Adagrad, optim.Adadelta,
                  optim.Adam, optim.Adamax, optim.NAdam],
}

grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X, y)

print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Mejor: 0.722656 usando {'optimizer': <class 'torch.optim.adam.Adam'>}
0.665365 (0.032106) with: {'optimizer': <class 'torch.optim.sgd.SGD'>}
0.662760 (0.012075) with: {'optimizer': <class 'torch.optim.rmsprop.RMSprop'>}
0.662760 (0.023510) with: {'optimizer': <class 'torch.optim.adagrad.Adagrad'>}
0.523438 (0.120314) with: {'optimizer': <class 'torch.optim.adadelta.Adadelta'>}
0.722656 (0.033146) with: {'optimizer': <class 'torch.optim.adam.Adam'>}
0.713542 (0.026557) with: {'optimizer': <class 'torch.optim.adamax.Adamax'>}
0.697917 (0.028764) with: {'optimizer': <class 'torch.optim.nadam.NAdam'>}


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section6"></a>
# <font color="#004D7F" size=6>6. Ajustar la Tasa de Aprendizaje y el Momentum</font>

Este enfoque permite una exploración sistemática de los parámetros clave del SGD, facilitando la optimización del rendimiento del modelo.

- Preselección común de algoritmo de optimización: SGD (Stochastic Gradient Descent)

  - Bien estudiado y comprendido

- Parámetros a optimizar en SGD:
  1. Tasa de aprendizaje:

     - Controla la magnitud de actualización de pesos por batch

  2. Momentum:

     - Influencia de la actualización anterior en la actual

- Uso en wrapper skorch:
  - Prefijo `optimizer__` para parámetros del optimizador

- Importancia de incluir número de épocas en la optimización:

  - Dependencia entre:

    - Aprendizaje por batch (tasa de aprendizaje)

    - Actualizaciones por época (tamaño del batch)

    - Número de épocas


In [11]:
param_grid = {
    'optimizer': [optim.SGD],
    'optimizer__lr': [0.001, 0.01, 0.1, 0.2, 0.3],
    'optimizer__momentum': [0.0, 0.2, 0.4, 0.6, 0.8, 0.9],
}

grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X, y)

print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Mejor: 0.684896 usando {'optimizer': <class 'torch.optim.sgd.SGD'>, 'optimizer__lr': 0.001, 'optimizer__momentum': 0.8}
0.665365 (0.018688) with: {'optimizer': <class 'torch.optim.sgd.SGD'>, 'optimizer__lr': 0.001, 'optimizer__momentum': 0.0}
0.666667 (0.012075) with: {'optimizer': <class 'torch.optim.sgd.SGD'>, 'optimizer__lr': 0.001, 'optimizer__momentum': 0.2}
0.665365 (0.036272) with: {'optimizer': <class 'torch.optim.sgd.SGD'>, 'optimizer__lr': 0.001, 'optimizer__momentum': 0.4}
0.654948 (0.050865) with: {'optimizer': <class 'torch.optim.sgd.SGD'>, 'optimizer__lr': 0.001, 'optimizer__momentum': 0.6}
0.684896 (0.007366) with: {'optimizer': <class 'torch.optim.sgd.SGD'>, 'optimizer__lr': 0.001, 'optimizer__momentum': 0.8}
0.567708 (0.153668) with: {'optimizer': <class 'torch.optim.sgd.SGD'>, 'optimizer__lr': 0.001, 'optimizer__momentum': 0.9}
0.475260 (0.177705) with: {'optimizer': <class 'torch.optim.sgd.SGD'>, 'optimizer__lr': 0.01, 'optimizer__momentum': 0.0}
0.550781 (0.140878) 

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section7"></a>
# <font color="#004D7F" size=6>7. Ajustar la Inicialización de Pesos de la Red</font>

Este ejemplo ofrece una visión general de cómo diferentes técnicas de inicialización pueden afectar el rendimiento de una red neuronal, aunque simplifica el proceso al usar un método único para todas las capas.

- Evolución de la inicialización de pesos:

  - Antes: Valores aleatorios pequeños
  
  - Ahora: Variedad de técnicas disponibles

- Referencia: Documentación de `torch.nn.init` en PyTorch

- Objetivo del ejemplo:
  - Explorar y ajustar diferentes técnicas de inicialización de pesos

- Enfoque:
  - Usar el mismo método de inicialización en todas las capas

  - No es lo ideal, pero simplifica el ejemplo

- Técnicas de inicialización a evaluar:
  - Uniforme
  - Normal
  - Xavier uniforme
  - Xavier normal
  - Kaiming uniforme
  - Kaiming normal
  - Ortogonal

- Consideraciones:
  - Idealmente, se usarían diferentes esquemas según la función de activación de cada capa

  - Este ejemplo simplifica usando un método único para todas las capas

- Ventajas de este enfoque:
  - Permite comparar el impacto de diferentes técnicas de inicialización
  
  - Proporciona una base para entender cómo la inicialización afecta el rendimiento del modelo

In [12]:
import torch.nn.init as init

class PimaClassifier(nn.Module):
    def __init__(self, weight_init=init.xavier_uniform_):
        super().__init__()
        self.layer = nn.Linear(8, 12)
        self.act = nn.ReLU()
        self.output = nn.Linear(12, 1)
        self.prob = nn.Sigmoid()
        # inicialización explícita de los pesos
        weight_init(self.layer.weight)
        weight_init(self.output.weight)

    def forward(self, x):
        x = self.act(self.layer(x))
        x = self.prob(self.output(x))
        return x

Esta modificación proporciona una manera flexible de controlar la inicialización de pesos en modelos PyTorch, facilitando la experimentación y optimización.

- Inicialización de pesos en PyTorch:
  - Es implícita

  - Requiere lógica personalizada para inicialización explícita

- Modificaciones necesarias en la clase del modelo:
  - Añadir argumento `weight_init` a la clase `PimaClassifier`

  - Este argumento espera un inicializador de `torch.nn.init`

- Implementación en la clase:
  - Se aplica `weight_init` a las capas lineales si se proporciona
  
  - Permite flexibilidad en la elección del método de inicialización

- Uso en `GridSearchCV`:
  - Utilizar el prefijo `module__` para parámetros del modelo

  - Permite que `NeuralNetClassifier` enrute correctamente los parámetros

In [13]:
import torch.nn.init as init

model = NeuralNetClassifier(
    PimaClassifier,
    criterion=nn.BCELoss,
    optimizer=optim.Adamax,
    max_epochs=100,
    batch_size=10,
    verbose=False
)

param_grid = {
    'module__weight_init': [init.uniform_, init.normal_, init.zeros_,
                            init.xavier_normal_, init.xavier_uniform_,
                            init.kaiming_normal_, init.kaiming_uniform_]
}
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X, y)

print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) con: %r" % (mean, stdev, param))

Mejor: 0.705729 usando {'module__weight_init': <function xavier_uniform_ at 0x7a462ed19440>}
0.348958 (0.001841) con: {'module__weight_init': <function uniform_ at 0x7a462ed18e00>}
0.647135 (0.014731) con: {'module__weight_init': <function normal_ at 0x7a462ed18ea0>}
0.651042 (0.001841) con: {'module__weight_init': <function zeros_ at 0x7a462ed191c0>}
0.673177 (0.006639) con: {'module__weight_init': <function xavier_normal_ at 0x7a462ed194e0>}
0.705729 (0.016053) con: {'module__weight_init': <function xavier_uniform_ at 0x7a462ed19440>}
0.558594 (0.149462) con: {'module__weight_init': <function kaiming_normal_ at 0x7a462ed196c0>}
0.574219 (0.159249) con: {'module__weight_init': <function kaiming_uniform_ at 0x7a462ed19620>}



Los mejores resultados se lograron con el esquema de inicialización de pesos `Kaiming-uniform`, alcanzando un rendimiento de aproximadamente `70%`.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section8"></a>
# <font color="#004D7F" size=6>8. Ajustar la Función de Activación de las Neuronas</font>

Este ejemplo proporciona una base para entender cómo diferentes funciones de activación pueden afectar el rendimiento del modelo en un problema de clasificación binaria.

- Función de activación:
  - Controla la no linealidad de las neuronas

  - Determina cuándo se "activan" las neuronas

- Funciones de activación populares:
  - ReLU (Rectified Linear Unit): Más utilizada actualmente

  - Sigmoid y tanh: Populares en el pasado, aún útiles en ciertos casos

- Enfoque del ejemplo:
  - Evaluar diferentes funciones de activación disponibles en PyTorch

  - Aplicar solo en la capa oculta

  - Mantener sigmoid en la capa de salida (clasificación binaria)

- Implementación:
  - Función de activación como argumento del constructor del modelo

  - Uso del prefijo `module__` en GridSearchCV para parámetros del modelo

- Funciones de activación a evaluar:
  - ReLU
  - Tanh
  - Sigmoid
  - LeakyReLU

- Ventajas de este enfoque:
  - Permite comparar el rendimiento de diferentes funciones de activación

  - Ayuda a identificar la función más adecuada para el problema específico

- Limitaciones:
  - No considera la preparación óptima de datos para cada función

  - Mantiene fija la función de activación en la capa de salida

In [14]:
class PimaClassifier(nn.Module):
    def __init__(self, activation=nn.Identity):
        super().__init__()
        self.layer = nn.Linear(8, 12)
        self.act = activation()
        self.output = nn.Linear(12, 1)
        self.prob = nn.Sigmoid()
        init.kaiming_uniform_(self.layer.weight)
        init.kaiming_uniform_(self.output.weight)

    def forward(self, x):
        x = self.act(self.layer(x))
        x = self.prob(self.output(x))
        return x

model = NeuralNetClassifier(
    PimaClassifier,
    criterion=nn.BCELoss,
    optimizer=optim.Adamax,
    max_epochs=100,
    batch_size=10,
    verbose=False
)

param_grid = {
    'module__activation': [nn.Identity, nn.ReLU, nn.ELU, nn.ReLU6, nn.GELU,
                           nn.Softplus, nn.Softsign, nn.Tanh, nn.Sigmoid, nn.Hardsigmoid]
}
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X, y)

print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) con: %r" % (mean, stdev, param))

Mejor: 0.671875 usando {'module__activation': <class 'torch.nn.modules.activation.ELU'>}
0.654948 (0.031948) con: {'module__activation': <class 'torch.nn.modules.linear.Identity'>}
0.557292 (0.148449) con: {'module__activation': <class 'torch.nn.modules.activation.ReLU'>}
0.671875 (0.038801) con: {'module__activation': <class 'torch.nn.modules.activation.ELU'>}
0.611979 (0.038450) con: {'module__activation': <class 'torch.nn.modules.activation.ReLU6'>}
0.656250 (0.024080) con: {'module__activation': <class 'torch.nn.modules.activation.GELU'>}
0.559896 (0.150288) con: {'module__activation': <class 'torch.nn.modules.activation.Softplus'>}
0.651042 (0.021710) con: {'module__activation': <class 'torch.nn.modules.activation.Softsign'>}
0.671875 (0.022326) con: {'module__activation': <class 'torch.nn.modules.activation.Tanh'>}
0.669271 (0.009744) con: {'module__activation': <class 'torch.nn.modules.activation.Sigmoid'>}
0.634115 (0.025780) con: {'module__activation': <class 'torch.nn.modules

Los resultados muestran que la función de activación `Softplus` obtuvo los mejores resultados, con un accuracy de aproximadamente `70%`.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section9"></a>
# <font color="#004D7F" size=6>9. Ajustar la Regularización Dropout</font>

Este enfoque sistemático permite optimizar la regularización del modelo, mejorando su capacidad de generalización y reduciendo el riesgo de sobreajuste.

- Objetivo: Ajustar la tasa de dropout para regularización
  - Limitar el sobreajuste

  - Mejorar la capacidad de generalización del modelo

- Recomendación: Combinar dropout con restricción de peso

  - Uso de restricción de norma máxima (`MaxNorm`)

  - Implementación en la función de paso hacia adelante (`forward`)

- Parámetros a ajustar:
  1. Porcentaje de dropout

  2. Valor de la restricción de norma máxima (MaxNorm)

- Rangos de valores a probar:

  - Dropout: 0.0 a 0.9 (1.0 no es adecuado)

  - MaxNorm: 0 a 5

- Proceso de evaluación:
  - Probar combinaciones de dropout y MaxNorm

  - Evaluar rendimiento para cada combinación

- Consideraciones:
  - El dropout 0.0 equivale a no aplicar dropout

  - MaxNorm 0 significa no aplicar restricción de norma máxima

- Ventajas de este enfoque:
  - Permite encontrar el balance óptimo entre regularización y rendimiento

  - Ayuda a prevenir el sobreajuste de manera más efectiva

- Implementación:
  - Utilizar bucles anidados o GridSearchCV para evaluar combinaciones
  
  - Registrar rendimiento para cada configuración

In [15]:
class PimaClassifier(nn.Module):
    def __init__(self, dropout_rate=0.5, weight_constraint=1.0):
        super().__init__()
        self.layer = nn.Linear(8, 12)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.output = nn.Linear(12, 1)
        self.prob = nn.Sigmoid()
        self.weight_constraint = weight_constraint
        init.kaiming_uniform_(self.layer.weight)
        init.kaiming_uniform_(self.output.weight)

    def forward(self, x):
        # restricción de norma máxima (max-norm) sobre los pesos de la capa oculta
        with torch.no_grad():
            norm = self.layer.weight.norm(2, dim=0, keepdim=True).clamp(min=self.weight_constraint / 2)
            desired = torch.clamp(norm, max=self.weight_constraint)
            self.layer.weight *= (desired / norm)
        x = self.act(self.layer(x))
        x = self.dropout(x)
        x = self.prob(self.output(x))
        return x

model = NeuralNetClassifier(
    PimaClassifier,
    criterion=nn.BCELoss,
    optimizer=optim.Adamax,
    max_epochs=100,
    batch_size=10,
    verbose=False
)

param_grid = {
    'module__weight_constraint': [1.0, 2.0, 3.0, 4.0, 5.0],
    'module__dropout_rate': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
}

grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X, y)

print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) con: %r" % (mean, stdev, param))

Mejor: 0.713542 usando {'module__dropout_rate': 0.2, 'module__weight_constraint': 2.0}
0.692708 (0.027126) con: {'module__dropout_rate': 0.0, 'module__weight_constraint': 1.0}
0.460938 (0.160204) con: {'module__dropout_rate': 0.0, 'module__weight_constraint': 2.0}
0.653646 (0.004872) con: {'module__dropout_rate': 0.0, 'module__weight_constraint': 3.0}
0.670573 (0.008027) con: {'module__dropout_rate': 0.0, 'module__weight_constraint': 4.0}
0.486979 (0.194277) con: {'module__dropout_rate': 0.0, 'module__weight_constraint': 5.0}
0.692708 (0.014731) con: {'module__dropout_rate': 0.1, 'module__weight_constraint': 1.0}
0.703125 (0.026107) con: {'module__dropout_rate': 0.1, 'module__weight_constraint': 2.0}
0.651042 (0.051560) con: {'module__dropout_rate': 0.1, 'module__weight_constraint': 3.0}
0.580729 (0.165431) con: {'module__dropout_rate': 0.1, 'module__weight_constraint': 4.0}
0.710938 (0.014616) con: {'module__dropout_rate': 0.1, 'module__weight_constraint': 5.0}
0.696615 (0.012890) con



Los resultados muestran que una tasa de `dropout` del `10%` y una restricción de norma máxima de `2.0` obtuvieron la mejor precisión, alrededor del `70%`.

Las líneas dentro del bloque `with torch.no_grad()` en tu función `forward` están relacionadas con la **normalización de los pesos** de la capa, y se ejecutan sin calcular gradientes. Esto es útil cuando deseas realizar operaciones que no afecten el proceso de retropropagación del modelo, como la actualización manual o la restricción de los pesos.

**Desglose de las líneas dentro del `with torch.no_grad()`**:

1. `norm = self.layer.weight.norm(2, dim=0, keepdim=True).clamp(min=self.weight_constraint / 2)`:

   - Propósito: Esta línea calcula la norma $L_2$ (norma euclidiana) de los pesos de la capa (`self.layer.weight`) a lo largo del eje 0 (dimensión de las características).

   - Detalles:

     - `self.layer.weight.norm(2, dim=0)`: Calcula la norma $L_2$ de los pesos a lo largo del eje 0.

     - `keepdim=True`: Mantiene las dimensiones originales después de la operación.

     - `.clamp(min=self.weight_constraint / 2)`: Restringe el valor mínimo de la norma a `self.weight_constraint / 2`. Si algún valor es menor que este límite, se ajusta al mínimo permitido.

2. `desired = torch.clamp(norm, max=self.weight_constraint)`:
   - Propósito: Esta línea ajusta el valor máximo permitido para la norma de los pesos.

   - Detalles:

     - `torch.clamp(norm, max=self.weight_constraint)`: Restringe el valor máximo de la norma calculada en la línea anterior a `self.weight_constraint`. Si algún valor excede este límite, se ajusta al máximo permitido.

3. `self.layer.weight *= (desired / norm)`:
   - Propósito: Esta línea ajusta los pesos de la capa para que cumplan con las restricciones impuestas por los valores calculados en las líneas anteriores.

   - Detalles:

     - Los pesos actuales (`self.layer.weight`) se multiplican por el factor correspondiente a $\frac{\text{desired}}{\text{norm}}$, lo que asegura que la norma de los pesos esté dentro del rango especificado por `self.weight_constraint`.

**¿Por qué usar `torch.no_grad()`?**

- El bloque `with torch.no_grad()` desactiva el cálculo automático de gradientes durante estas operaciones. Esto es importante porque:

  - Estas modificaciones a los pesos no deben afectar el proceso de retropropagación.

  - Evita que PyTorch almacene información innecesaria para calcular gradientes, lo que ahorra memoria y mejora el rendimiento.

**Resumen del bloque `with torch.no_grad()`**:

- Se calcula y ajusta la norma $L_2$ de los pesos de la capa para asegurarse de que estén dentro de un rango específico definido por `self.weight_constraint`.

- Los pesos se normalizan sin afectar el cálculo de gradientes durante el entrenamiento.
  
Este tipo de operación es común cuando se desea imponer restricciones sobre los valores o magnitudes de los parámetros (como regularización o normalización) sin interferir con el aprendizaje del modelo.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section10"></a>
# <font color="#004D7F" size=6>10. Ajustar el Número de Neuronas en la Capa Oculta</font>

Este enfoque sistemático permite encontrar el balance óptimo entre la capacidad del modelo y su capacidad de generalización, considerando la interacción con otros parámetros de entrenamiento.

- Importancia del número de neuronas en una capa:
  - Controla la capacidad representacional de la red

  - Influye en el punto específico de la topología

- Ajuste del número de neuronas:
  - Rango de prueba: 1 a 30 neuronas
  - Incrementos: Pasos de 5

- Consideraciones importantes:
  - Redes más grandes requieren más entrenamiento

  - Optimización conjunta recomendada:

    - Número de neuronas

    - Tamaño del lote (batch size)

    - Número de épocas

- Riesgos:
  - Pocas neuronas: Posible subajuste

  - Demasiadas neuronas: Posible sobreajuste

- Método de optimización:
  - Usar validación cruzada para evaluar rendimiento
  
  - Seleccionar el número de neuronas que maximiza el rendimiento promedio

- Herramienta sugerida:
  - GridSearchCV para búsqueda sistemática de hiperparámetros

In [16]:
class PimaClassifier(nn.Module):
    def __init__(self, n_neurons=12):
        super().__init__()
        self.layer = nn.Linear(8, n_neurons)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.output = nn.Linear(n_neurons, 1)
        self.prob = nn.Sigmoid()
        self.weight_constraint = 2.0
        init.kaiming_uniform_(self.layer.weight)
        init.kaiming_uniform_(self.output.weight)

    def forward(self, x):
        with torch.no_grad():
            norm = self.layer.weight.norm(2, dim=0, keepdim=True).clamp(min=self.weight_constraint / 2)
            desired = torch.clamp(norm, max=self.weight_constraint)
            self.layer.weight *= (desired / norm)
        x = self.act(self.layer(x))
        x = self.dropout(x)
        x = self.prob(self.output(x))
        return x

model = NeuralNetClassifier(
    PimaClassifier,
    criterion=nn.BCELoss,
    optimizer=optim.Adamax,
    max_epochs=100,
    batch_size=10,
    verbose=False
)

param_grid = {
    'module__n_neurons': [1, 5, 10, 15, 20, 25, 30]
}
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X, y)

print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) con: %r" % (mean, stdev, param))

Mejor: 0.708333 usando {'module__n_neurons': 15}
0.670573 (0.036966) con: {'module__n_neurons': 1}
0.675781 (0.019918) con: {'module__n_neurons': 5}
0.701823 (0.034104) con: {'module__n_neurons': 10}
0.708333 (0.024150) con: {'module__n_neurons': 15}
0.588542 (0.167843) con: {'module__n_neurons': 20}
0.704427 (0.032106) con: {'module__n_neurons': 25}
0.595052 (0.176354) con: {'module__n_neurons': 30}


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section11"></a>
# <font color="#004D7F" size=6>11. Consejos para la Optimización de Hiperparámetros</font>

Esta sección enumera algunos consejos útiles a considerar al ajustar los hiperparámetros de tu red neuronal.
- **Validación Cruzada k-Fold**: Los resultados de los ejemplos en este capítulo muestran cierta variación. 

    - Se utilizó una validación cruzada por defecto de 3, pero tal vez k = 5 o k = 10 sería más estable. 
    
    - Elige cuidadosamente tu configuración de validación cruzada para asegurar que tus resultados sean estables.

- **Revisa el grid search Completa**: No te centres únicamente en el mejor resultado; revisa toda el grid search de resultados y busca tendencias que apoyen las decisiones de configuración. 

    - Por supuesto, habrá más combinaciones y tomará más tiempo evaluarlas.

- **Paraleliza**: Usa todos tus núcleos si puedes. Las redes neuronales son lentas de entrenar, y a menudo queremos probar muchos parámetros diferentes. 
    
    - Considera ejecutarlo en una plataforma en la nube, como AWS.

- **Usa una Muestra de tu Conjunto de Datos**: Dado que las redes son lentas de entrenar, intenta entrenarlas en una muestra más pequeña de tu conjunto de datos de entrenamiento, solo para tener una idea de las direcciones generales de los parámetros en lugar de configuraciones óptimas.

- **Comienza con grid search Amplias**: Comienza con grid search de ajuste de gran tamaño y luego profundiza en grid search más finas una vez que puedas reducir el alcance.

- **No Transfieras Resultados**: Los resultados son generalmente específicos para el problema. 

    - Trata de evitar configuraciones favoritas en cada nuevo problema que encuentres. 
    
    - Es poco probable que los resultados óptimos que descubras en un problema se transfieran a tu próximo proyecto. 
    
    - En su lugar, busca tendencias más amplias, como el número de capas o relaciones entre parámetros.

- **La Reproducibilidad es un Problema**: Aunque establezcamos la semilla para el generador de números aleatorios en NumPy, los resultados no son 100% reproducibles. 
    
    - Hay más aspectos de la reproducibilidad al buscar en la grilla modelos de PyTorch envueltos de los que se presentan en este capítulo.

<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>